In [1]:
import pandas as pd
from blended_learning.config.settings import settings
from blended_learning.utils.decorator import execution_time


In [2]:
df = pd.read_csv(settings.path['output_cluster_path'])

In [3]:
columns = [
    'student_id',
    'open_strengths',
    'open_challenges_suggestions',
    'student_segment',
    'student_segment_label',
    'cluster_label'
]
df_nlp = df[columns].copy()
print(f"NLP dataframe shape: {df_nlp.shape}")
display(df_nlp.head(n=2))

NLP dataframe shape: (588, 6)


,student_id,open_strengths,open_challenges_suggestions,student_segment,student_segment_label,cluster_label
0,e20210528,NaN,NaN,1,Highly Engaged (Active) Learners,Moderately Engaged (Passive) Learners
1,e20210686,Nothing,Nothing,2,Moderately Engaged (Passive) Learners,Highly Engaged (Active) Learners


In [4]:
print(f"Before : Length {len(df_nlp)}")

cols = ['open_strengths', 'open_challenges_suggestions']

mask = (
    df_nlp[cols].notna().all(axis=1) &
    (df_nlp[cols].astype(str).apply(lambda x: x.str.strip() != "")).all(axis=1)
)

df_nlp = df_nlp[mask]

print(f"After : Length {len(df_nlp)}")

Before : Length 588
After : Length 567


In [5]:
import re
import pandas as pd

KHMER_REGEX = re.compile(r'[\u1780-\u17FF]')
EN_REGEX = re.compile(r'[A-Za-z]')

def detect_lang_mixed(text):
    if pd.isna(text):
        return "empty"
    
    text = str(text).strip()
    if text == "":
        return "empty"
    
    has_kh = bool(KHMER_REGEX.search(text))
    has_en = bool(EN_REGEX.search(text))
    
    if has_kh and has_en:
        return "mixed"
    elif has_kh:
        return "kh"
    elif has_en:
        return "en"
    else:
        return "other"


# Create language label columns in df_nlp, not df
for col in cols:
    df_nlp[f"{col}_lang"] = df_nlp[col].apply(detect_lang_mixed)


# Print summary counts + percentages per column
all_langs = []

for col in cols:
    cleaned_mask = df_nlp[col].notna() & (df_nlp[col].astype(str).str.strip() != "")
    langs = df_nlp.loc[cleaned_mask, f"{col}_lang"]
    all_langs.append(langs)
    
    counts = langs.value_counts().reindex(["kh", "mixed", "en", "other"], fill_value=0)
    perc = (counts / counts.sum() * 100).round(1)
    
    summary = pd.DataFrame({
        "count": counts,
        "percent (%)": perc
    })
    
    translation_needed = counts["kh"] + counts["mixed"]
    translation_pct = round((translation_needed / counts.sum()) * 100, 1) if counts.sum() > 0 else 0
    
    print(f"\n{'='*60}")
    print(f"SUMMARY FOR: {col}")
    print(f"{'='*60}")
    print(summary)
    print(f"\nTotal needing translation (kh + mixed): {translation_needed} ({translation_pct}%)")


# Aggregate summary across both text columns
all_langs_combined = pd.concat(all_langs, axis=0)

agg_counts = all_langs_combined.value_counts().reindex(["kh", "mixed", "en", "other"], fill_value=0)
agg_perc = (agg_counts / agg_counts.sum() * 100).round(1)

agg_summary = pd.DataFrame({
    "count": agg_counts,
    "percent (%)": agg_perc
})

agg_translation_needed = agg_counts["kh"] + agg_counts["mixed"]
agg_translation_pct = round((agg_translation_needed / agg_counts.sum()) * 100, 1)

print(f"\n{'='*60}")
print("OVERALL LANGUAGE SUMMARY")
print(f"{'='*60}")
print(agg_summary)
print(f"\nOverall needing translation: {agg_translation_needed} ({agg_translation_pct}%)")


SUMMARY FOR: open_strengths
                     count  percent (%)
open_strengths_lang                    
kh                      24          4.2
mixed                    2          0.4
en                     535         94.4
other                    6          1.1

Total needing translation (kh + mixed): 26 (4.6%)

SUMMARY FOR: open_challenges_suggestions
                                  count  percent (%)
open_challenges_suggestions_lang                    
kh                                   23          4.1
mixed                                 5          0.9
en                                  533         94.0
other                                 6          1.1

Total needing translation (kh + mixed): 28 (4.9%)



OVERALL LANGUAGE SUMMARY
       count  percent (%)
kh        47          4.1
mixed      7          0.6
en      1068         94.2
other     12          1.1

Overall needing translation: 54 (4.8%)


Translate Khmer and mixed responses

In [6]:
from blended_learning.nlp.translate_kh_service import TranslateKHService

translator = TranslateKHService()
translation_cache = {}


def translate_if_needed(text, lang):
    if pd.isna(text):
        return ""

    text = str(text).strip()

    if text == "":
        return ""

    if lang not in ["kh", "mixed"]:
        return text

    if text in translation_cache:
        return translation_cache[text]

    translated = translator.translate_one(
        text=text,
        src_lang="kh",
        tgt_lang="eng"
    )

    translation_cache[text] = translated
    return translated

In [7]:
@execution_time
def translate_column(df_nlp, col):
    translated_col = f"{col}_en"

    df_nlp[translated_col] = df_nlp.apply(
        lambda row: translate_if_needed(
            row[col],
            row[f"{col}_lang"]
        ),
        axis=1
    )

    print(f"Finished translating: {col}")
    return df_nlp


for col in cols:
    df_nlp = translate_column(df_nlp, col)

Finished translating: open_strengths
translate_column executed in 0.44 min (26.18 sec)
Finished translating: open_challenges_suggestions
translate_column executed in 0.34 min (20.55 sec)


In [8]:
display(
    df_nlp[
        [
            "open_strengths",
            "open_strengths_lang",
            "open_strengths_en",
            "open_challenges_suggestions",
            "open_challenges_suggestions_lang",
            "open_challenges_suggestions_en",
        ]
    ].head(10)
)

,open_strengths,open_strengths_lang,open_strengths_en,open_challenges_suggestions,open_challenges_suggestions_lang,open_challenges_suggestions_en
1,Nothing,en,Nothing,Nothing,en,Nothing
2,"Very good, excellent",en,"Very good, excellent","No big challenge, i’m the best",en,"No big challenge, i’m the best"
3,Will try hard,en,Will try hard,Lack of self-discipline,en,Lack of self-discipline
4,getting more experience,en,getting more experience,Discipline on daily studying,en,Discipline on daily studying
5,Student could retrieve the lesson once they wa...,en,Student could retrieve the lesson once they wa...,Add Ai assistance,en,Add Ai assistance
6,Innovate yourself with the technology world,en,Innovate yourself with the technology world,Lack of resources and misinformation,en,Lack of resources and misinformation
7,Blended learning has many positive aspects. It...,en,Blended learning has many positive aspects. It...,The biggest challenges of blended learning are...,en,The biggest challenges of blended learning are...
8,Give a chance for students who prefer both onl...,en,Give a chance for students who prefer both onl...,"Internet issues, technical problems, lack of i...",en,"Internet issues, technical problems, lack of i..."
10,Give me time to do my stuff,en,Give me time to do my stuff,Probably the internet connection,en,Probably the internet connection
11,Agree,en,Agree,Agree,en,Agree


In [9]:
output_path = settings.root / "data" / "processed" / "nlp_translated_responses.csv"

df_nlp.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved translated NLP dataset to: {output_path}")

Saved translated NLP dataset to: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\nlp_translated_responses.csv
